In [ ]:
# NORTHSTAR -- Tower 33 Evidence Chains QA for Solace Browser
# DNA: evidence(chain) = capture(screenshot) x hash(sha256) x chain(append) x retain(90d) x query(search) x export(bundle) x tamper(detect)
# Probes evidence capture, SHA-256 hashing, hash chaining, retention, export, tamper detection
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "evidence-t33-qa"
NOTEBOOK_PATH = "notebooks/qa/19-evidence-t33-qa.ipynb"
TOWER = 33
TOWER_NAME = "Tower of Evidence Chains"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
EVIDENCE = SRC / "evidence"
AUDIT = SRC / "audit"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

In [ ]:
# F1 -- Rachel Kim: Screenshot Capture (Product Liability Lawyer)
# Q: Does browser capture screenshots at every critical step?

# Probe: capture pipeline has screenshot capability
capture_code = read_file(SRC / "capture_pipeline.py")
has_screenshot = bool(re.search(r'screenshot|capture.*image|png|save.*image|take.*photo', capture_code, re.IGNORECASE))
record("F1-P1", "Capture pipeline supports screenshot capture", has_screenshot,
       f"{len(capture_code)} bytes")

# Probe: screenshots include metadata (URL, timestamp)
has_metadata = bool(re.search(r'url|timestamp|metadata|datetime|step', capture_code, re.IGNORECASE))
record("F1-P2", "Screenshots include metadata (URL, timestamp)", has_metadata)

# Probe: capture test exists
capture_test = TESTS / "test_capture_pipeline.py"
has_test = capture_test.exists() and capture_test.stat().st_size > 100
record("F1-P3", "Capture pipeline test exists", has_test)

# Probe: evidence upload module exists
upload_code = read_file(SRC / "evidence_upload.py")
has_upload = len(upload_code) > 100
record("F1-P4", "Evidence upload module exists", has_upload,
       f"{len(upload_code)} bytes")

In [ ]:
# F2 -- Dmitri Volkov: SHA-256 Hashing (Blockchain Architect)
# Q: Is every artifact SHA-256 hashed? Can I verify independently?

# Probe: audit chain module uses SHA-256
chain_code = read_file(AUDIT / "chain.py")
has_sha256 = bool(re.search(r'sha256|hashlib\.sha|sha_256', chain_code, re.IGNORECASE))
record("F2-P1", "Audit chain uses SHA-256 hashing", has_sha256,
       f"{len(chain_code)} bytes")

# Probe: hash chain links previous hash to current
has_chain = bool(re.search(r'previous|prev.*hash|chain|link|parent', chain_code, re.IGNORECASE))
record("F2-P2", "Hash chain links records with previous hash", has_chain)

# Probe: evidence summary formatter exists
summary_fmt = read_file(EVIDENCE / "summary_formatter.py")
has_fmt = len(summary_fmt) > 100
record("F2-P3", "Evidence summary formatter exists", has_fmt,
       f"{len(summary_fmt)} bytes")

# Probe: evidence chain integration test
chain_test = TESTS / "test_evidence_chain_integration.py"
has_chain_test = chain_test.exists() and chain_test.stat().st_size > 100
record("F2-P4", "Evidence chain integration test exists", has_chain_test)

In [ ]:
# F3 -- Chain Append (immutable)
# Q: Is the evidence chain append-only? Can records be modified?

# Probe: chain is append-only (no delete/modify in chain.py)
chain_code = read_file(AUDIT / "chain.py")
has_append = bool(re.search(r'append|add|insert|seal', chain_code, re.IGNORECASE))
no_delete = not bool(re.search(r'delete|remove|pop|truncate|overwrite', chain_code, re.IGNORECASE))
record("F3-P1", "Chain supports append operations", has_append)
record("F3-P2", "Chain has no delete/modify operations", no_delete,
       "append-only verified" if no_delete else "delete/modify found in chain")

# Probe: ALCOA compliance module exists (attribution, legibility, contemporaneous, original, accurate)
alcoa_code = read_file(AUDIT / "alcoa.py")
has_alcoa = len(alcoa_code) > 100
record("F3-P3", "ALCOA compliance module exists", has_alcoa,
       f"{len(alcoa_code)} bytes")

In [ ]:
# F4 -- Retention (90 days)
# Q: Does the system retain evidence for 90 days?

# Probe: retention module exists
retention_code = read_file(AUDIT / "retention.py")
has_retention = bool(re.search(r'retain|retention|ttl|expire|days|purge|cleanup', retention_code, re.IGNORECASE))
record("F4-P1", "Retention module exists with policy logic", has_retention,
       f"{len(retention_code)} bytes")

# Probe: retention period is defined (look for 90 or configurable)
has_90d = bool(re.search(r'90|retention.*day|day.*retention|RETENTION', retention_code, re.IGNORECASE))
record("F4-P2", "Retention period defined (90 days or configurable)", has_90d)

# Probe: audit test exists
audit_test = TESTS / "test_audit.py"
has_audit_test = audit_test.exists() and audit_test.stat().st_size > 100
record("F4-P3", "Audit test exists", has_audit_test)

In [ ]:
# F5 -- Query/Search + F6 -- Export
# Q: Can evidence be searched? Can it be exported as a bundle?

# Probe: evidence summary allows searching/formatting
summary_code = read_file(EVIDENCE / "summary_formatter.py")
has_search = bool(re.search(r'search|query|filter|find|format|render', summary_code, re.IGNORECASE))
record("F5-P1", "Evidence summary supports formatting/searching", has_search,
       f"{len(summary_code)} bytes")

# Probe: evidence summary test exists
fmt_test = TESTS / "test_evidence_summary_formatter.py"
has_fmt_test = fmt_test.exists() and fmt_test.stat().st_size > 100
record("F5-P2", "Evidence summary formatter test exists", has_fmt_test)

# Probe: evidence has export/bundle capability
all_evidence_code = ""
for f in list(EVIDENCE.glob("*.py")) + list(AUDIT.glob("*.py")):
    all_evidence_code += f.read_text(encoding='utf-8', errors='replace')
has_export = bool(re.search(r'export|bundle|zip|download|archive', all_evidence_code, re.IGNORECASE))
record("F5-P3", "Evidence supports export/bundle", has_export)

In [ ]:
# F7 -- Tamper Detection
# NEGATIVE SPACE (P16) -- Evidence integrity gaps

# Probe: tamper detection via hash verification
has_verify = bool(re.search(r'verify|validate|tamper|integrity|check.*hash', all_evidence_code, re.IGNORECASE))
record("F7-P1", "Evidence has tamper detection/hash verification", has_verify)

# Probe: no bare excepts in evidence modules (Fallback Ban)
evidence_files = list(EVIDENCE.glob("*.py")) + list(AUDIT.glob("*.py"))
bare_excepts = 0
for ef in evidence_files:
    content = ef.read_text(encoding='utf-8', errors='replace')
    bare_excepts += len(re.findall(r'^\s*except\s*:', content, re.MULTILINE))
    bare_excepts += len(re.findall(r'^\s*except\s+Exception\s*:', content, re.MULTILINE))
record("NS-P1", "No bare excepts in evidence/audit (Fallback Ban)",
       bare_excepts == 0, f"{bare_excepts} bare excepts across {len(evidence_files)} files")

# Probe: OAuth3 evidence module links OAuth3 decisions to evidence chain
oauth3_evidence = read_file(SRC / "oauth3" / "evidence.py")
has_oauth3_ev = len(oauth3_evidence) > 100
record("NS-P2", "OAuth3 evidence module links consent to chain", has_oauth3_ev,
       f"{len(oauth3_evidence)} bytes")

# Probe: evidence __init__.py exports properly
ev_init = read_file(EVIDENCE / "__init__.py")
record("NS-P3", "Evidence package initialized", len(ev_init) > 0)

In [ ]:
# EVIDENCE SUMMARY -- Tower 33 Evidence Chains QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))